In [ ]:
# Week 3 Day 5 — Encoder Decoder
# Goal: Build encoder and decoder with skip connections

import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("✓ Setup complete")

Device: cuda
✓ Setup complete


In [ ]:
# Cell 2 — Encoder Block
# Convolution + ReLU + Pooling
# Makes image smaller, learns features

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels,
                               kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, padding=1)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        # Two convolutions + ReLU
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        # Save for skip connection BEFORE pooling
        skip = x
        # Pool to reduce size
        x = self.pool(x)
        return x, skip

# Test encoder block
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = EncoderBlock(in_channels=1, out_channels=16).to(device)

# Simulate CT slice
ct_input = torch.zeros(1, 1, 64, 64).to(device)
print(f"Input shape:        {ct_input.shape}")

# Forward pass
with torch.no_grad():
    output, skip = encoder(ct_input)

print(f"Output shape:       {output.shape}")
print(f"Skip connection:    {skip.shape}")
print(f"\nInput  [1,  1, 64, 64]")
print(f"Skip   [1, 16, 64, 64]  ← saved before pooling")
print(f"Output [1, 16, 32, 32]  ← after pooling, halved")

Input shape:        torch.Size([1, 1, 64, 64])
Output shape:       torch.Size([1, 16, 32, 32])
Skip connection:    torch.Size([1, 16, 64, 64])

Input  [1,  1, 64, 64]
Skip   [1, 16, 64, 64]  ← saved before pooling
Output [1, 16, 32, 32]  ← after pooling, halved


In [ ]:
# Cell 3 — Decoder Block
# Upsampling + Concatenation + Convolution
# Makes image larger, recovers spatial detail

import torch
import torch.nn as nn

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2,
                                     mode='bilinear',
                                     align_corners=True)
        self.conv1 = nn.Conv2d(in_channels * 2, out_channels,
                               kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, padding=1)
        self.relu  = nn.ReLU()

    def forward(self, x, skip):
        x = self.upsample(x)
        print(f"  After upsample:      {x.shape}")
        x = torch.cat([x, skip], dim=1)
        print(f"  After concatenation: {x.shape}")
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        print(f"  After convolutions:  {x.shape}")
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
decoder = DecoderBlock(in_channels=16, out_channels=16).to(device)

x    = torch.zeros(1, 16, 32, 32).to(device)
skip = torch.zeros(1, 16, 64, 64).to(device)

print(f"Input shape:     {x.shape}")
print(f"Skip shape:      {skip.shape}")
print(f"\nDecoder steps:")

with torch.no_grad():
    output = decoder(x, skip)

print(f"\nFinal output:    {output.shape}")

Input shape:     torch.Size([1, 16, 32, 32])
Skip shape:      torch.Size([1, 16, 64, 64])

Decoder steps:
  After upsample:      torch.Size([1, 16, 64, 64])
  After concatenation: torch.Size([1, 32, 64, 64])
  After convolutions:  torch.Size([1, 16, 64, 64])

Final output:    torch.Size([1, 16, 64, 64])


In [ ]:
# Cell 4 — Skip connections
# Showing how encoder and decoder connect

import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("── Skip connection demonstration ──")
print("Encoder saves feature maps at each level")
print("Decoder receives them at matching level")

# Simulate encoder outputs at 3 levels
skip1 = torch.zeros(1, 16, 64, 64).to(device)  # level 1
skip2 = torch.zeros(1, 32, 32, 32).to(device)  # level 2
skip3 = torch.zeros(1, 64, 16, 16).to(device)  # level 3

print(f"\nEncoder skip connections saved:")
print(f"Level 1: {skip1.shape}  ← largest, most detail")
print(f"Level 2: {skip2.shape}  ← medium")
print(f"Level 3: {skip3.shape}  ← smallest, most compressed")

# Simulate bottleneck output
bottleneck = torch.zeros(1, 128, 8, 8).to(device)
print(f"\nBottleneck: {bottleneck.shape}  ← most compressed")

print(f"\nDecoder receives skip connections:")
print(f"Level 3 decoder gets: {skip3.shape}")
print(f"Level 2 decoder gets: {skip2.shape}")
print(f"Level 1 decoder gets: {skip1.shape}")

print(f"\n── Concatenation at each level ──")

# Level 3 decoder
upsample = nn.Upsample(scale_factor=2,
                        mode='bilinear',
                        align_corners=True)

x = upsample(bottleneck)
print(f"\nBottleneck upsampled: {x.shape}")
x = torch.cat([x, skip3], dim=1)
print(f"After cat with skip3: {x.shape}")

x = upsample(x[:, :64])
print(f"\nUpsampled again:      {x.shape}")
x = torch.cat([x, skip2], dim=1)
print(f"After cat with skip2: {x.shape}")

x = upsample(x[:, :32])
print(f"\nUpsampled again:      {x.shape}")
x = torch.cat([x, skip1], dim=1)
print(f"After cat with skip1: {x.shape}")

print(f"\n✓ Skip connections successfully")
print(f"  carried detail from encoder to decoder")

── Skip connection demonstration ──
Encoder saves feature maps at each level
Decoder receives them at matching level

Encoder skip connections saved:
Level 1: torch.Size([1, 16, 64, 64])  ← largest, most detail
Level 2: torch.Size([1, 32, 32, 32])  ← medium
Level 3: torch.Size([1, 64, 16, 16])  ← smallest, most compressed

Bottleneck: torch.Size([1, 128, 8, 8])  ← most compressed

Decoder receives skip connections:
Level 3 decoder gets: torch.Size([1, 64, 16, 16])
Level 2 decoder gets: torch.Size([1, 32, 32, 32])
Level 1 decoder gets: torch.Size([1, 16, 64, 64])

── Concatenation at each level ──

Bottleneck upsampled: torch.Size([1, 128, 16, 16])
After cat with skip3: torch.Size([1, 192, 16, 16])

Upsampled again:      torch.Size([1, 64, 32, 32])
After cat with skip2: torch.Size([1, 96, 32, 32])

Upsampled again:      torch.Size([1, 32, 64, 64])
After cat with skip1: torch.Size([1, 48, 64, 64])

✓ Skip connections successfully
  carried detail from encoder to decoder


In [1]:
# Cell 5 — Complete Encoder-Decoder FIXED

import torch
import torch.nn as nn

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels,
                               kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, padding=1)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x    = self.relu(self.conv1(x))
        x    = self.relu(self.conv2(x))
        skip = x
        x    = self.pool(x)
        return x, skip

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2,
                                     mode='bilinear',
                                     align_corners=True)
        # in_channels + skip_channels after concatenation
        self.conv1 = nn.Conv2d(in_channels + skip_channels,
                               out_channels,
                               kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, padding=1)
        self.relu  = nn.ReLU()

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x

class SimpleEncoderDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = EncoderBlock(1,  16)
        self.enc2 = EncoderBlock(16, 32)
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )
        # Decoder
        # dec2: bottleneck=64 + skip2=32 = 96 input channels
        self.dec2 = DecoderBlock(in_channels=64,
                                  skip_channels=32,
                                  out_channels=32)
        # dec1: dec2 output=32 + skip1=16 = 48 input channels
        self.dec1 = DecoderBlock(in_channels=32,
                                  skip_channels=16,
                                  out_channels=16)
        # Output
        self.output  = nn.Conv2d(16, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        print(f"Input:          {x.shape}")

        # Encoder
        x, skip1 = self.enc1(x)
        print(f"After enc1:     {x.shape}  skip1: {skip1.shape}")

        x, skip2 = self.enc2(x)
        print(f"After enc2:     {x.shape}  skip2: {skip2.shape}")

        # Bottleneck
        x = self.bottleneck(x)
        print(f"Bottleneck:     {x.shape}")

        # Decoder
        x = self.dec2(x, skip2)
        print(f"After dec2:     {x.shape}")

        x = self.dec1(x, skip1)
        print(f"After dec1:     {x.shape}")

        # Output
        x = self.sigmoid(self.output(x))
        print(f"Output:         {x.shape}")

        return x

# Create and test model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = SimpleEncoderDecoder().to(device)

ct_input = torch.zeros(1, 1, 64, 64).to(device)

print(f"Device: {device}\n")
with torch.no_grad():
    output = model(ct_input)

print(f"\n── Summary ──")
print(f"Input  shape: [1, 1, 64, 64]")
print(f"Output shape: {output.shape}")
print(f"Each pixel = tumour probability 0-1")
print(f"✓ Complete encoder-decoder working")

Device: cuda

Input:          torch.Size([1, 1, 64, 64])
After enc1:     torch.Size([1, 16, 32, 32])  skip1: torch.Size([1, 16, 64, 64])
After enc2:     torch.Size([1, 32, 16, 16])  skip2: torch.Size([1, 32, 32, 32])
Bottleneck:     torch.Size([1, 64, 16, 16])
After dec2:     torch.Size([1, 32, 32, 32])
After dec1:     torch.Size([1, 16, 64, 64])
Output:         torch.Size([1, 1, 64, 64])

── Summary ──
Input  shape: [1, 1, 64, 64]
Output shape: torch.Size([1, 1, 64, 64])
Each pixel = tumour probability 0-1
✓ Complete encoder-decoder working
